# HealthSpark — ML Pipeline Walkthrough

Step-by-step walkthrough of the PySpark MLlib pipeline for **30-day readmission prediction**.

This notebook mirrors what the production pipeline does (`src/pipeline/ml_pipeline.py`),
but pauses at each stage to inspect schemas, show intermediate outputs, and explain decisions.

**Goal**: Predict whether a patient will be readmitted within 30 days of discharge.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from src.utils.spark_session import get_spark_session
from src.pipeline.ingestion import CLAIMS_SCHEMA, PATIENTS_SCHEMA

spark = get_spark_session(app_name='HealthSpark-ML-Notebook')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print(f'Spark version: {spark.version}')

## Step 1: Load Data

We load the raw claims CSV and inspect the schema. Using explicit `StructType` schemas
avoids Spark's schema inference pass (which reads the entire file just to guess types).

In [ ]:
claims_df = spark.read.option('header', 'true').option('dateFormat', 'yyyy-MM-dd').schema(CLAIMS_SCHEMA).csv('../data/raw/claims.csv')
print(f'Records: {claims_df.count():,}')
claims_df.printSchema()
claims_df.show(5, truncate=False)

## Step 2: Feature Engineering

We engineer 15+ features using PySpark-native operations. Each feature is designed
to capture a different dimension of readmission risk.

In [ ]:
from src.pipeline.transforms import add_rolling_claim_cost, add_lag_lead_features
from src.pipeline.feature_engineering import (
    add_diagnosis_risk_score, add_los_ratio, add_provider_denial_rate,
    add_payer_approval_rate, add_age_bucket, add_cost_features, add_comorbidity_index
)

# Apply feature transforms
features_df = claims_df
features_df = add_rolling_claim_cost(features_df)
features_df = add_lag_lead_features(features_df)
features_df = add_diagnosis_risk_score(features_df)
features_df = add_los_ratio(features_df)
features_df = add_provider_denial_rate(features_df)
features_df = add_payer_approval_rate(features_df)
features_df = add_age_bucket(features_df)
features_df = add_cost_features(features_df)
features_df = add_comorbidity_index(features_df)

# Fill nulls for temporal features
features_df = features_df.fillna({
    'days_since_last_claim': -1, 'prev_denial_flag': 0, 'prev_claim_amount': 0.0,
    'rolling_cost_90d': 0.0, 'claim_count_90d': 0, 'claim_sequence': 1,
    'provider_denial_rate': 0.0, 'provider_claim_volume': 0,
    'payer_approval_rate': 0.85, 'patient_avg_claim': 0.0,
    'patient_total_claims': 1, 'cost_ratio_to_patient_avg': 1.0,
})

features_df.cache()
print(f'Engineered features: {len(features_df.columns)} columns')
print('\nSchema after feature engineering:')
features_df.printSchema()

## Step 3: Train/Test Split

80/20 random split with a fixed seed. We check the class balance to ensure
both splits have similar readmission rates.

In [ ]:
features_df = features_df.withColumn('readmission_30day', F.col('readmission_30day').cast('integer'))
train_df, test_df = features_df.randomSplit([0.8, 0.2], seed=42)

print(f'Train: {train_df.count():,} | Test: {test_df.count():,}')
print(f'Train readmission rate: {train_df.select(F.avg("readmission_30day")).first()[0]:.1%}')
print(f'Test readmission rate: {test_df.select(F.avg("readmission_30day")).first()[0]:.1%}')

## Step 4: Build the MLlib Pipeline

The pipeline chains together all preprocessing and modeling stages:

```
StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler → RandomForestClassifier
```

**Why a Pipeline object?** Serializing all stages together eliminates train/serve skew —
the exact same preprocessing runs at inference time.

In [ ]:
cat_cols = ['facility_type', 'insurance_type', 'gender', 'age_bucket']
indexed_cols = [f'{c}_idx' for c in cat_cols]
ohe_cols = [f'{c}_ohe' for c in cat_cols]

numeric_cols = [
    'claim_amount', 'paid_amount', 'length_of_stay', 'age',
    'comorbidity_count', 'diagnosis_risk_score', 'los_vs_expected_ratio',
    'provider_denial_rate', 'provider_claim_volume', 'payer_approval_rate',
    'patient_avg_claim', 'patient_total_claims', 'cost_ratio_to_patient_avg',
    'comorbidity_index', 'rolling_cost_90d', 'claim_count_90d',
    'days_since_last_claim', 'prev_denial_flag', 'prev_claim_amount', 'claim_sequence',
]

# Pipeline stages
indexers = [StringIndexer(inputCol=c, outputCol=i, handleInvalid='keep') for c, i in zip(cat_cols, indexed_cols)]
encoder = OneHotEncoder(inputCols=indexed_cols, outputCols=ohe_cols, dropLast=True)
assembler = VectorAssembler(inputCols=numeric_cols + ohe_cols, outputCol='raw_features', handleInvalid='skip')
scaler = StandardScaler(inputCol='raw_features', outputCol='scaled_features', withMean=False, withStd=True)
rf = RandomForestClassifier(featuresCol='scaled_features', labelCol='readmission_30day',
                            predictionCol='prediction', probabilityCol='probability',
                            rawPredictionCol='raw_prediction', seed=42)

pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler, rf])
print(f'Pipeline stages: {len(pipeline.getStages())}')
for i, stage in enumerate(pipeline.getStages()):
    print(f'  Stage {i}: {type(stage).__name__}')

## Step 5: Hyperparameter Tuning with CrossValidator

We search over `numTrees × maxDepth × maxBins` = 27 combinations with 3-fold CV.

**Note**: This takes several minutes in local mode. In Databricks, it would run
across the cluster with `parallelism` matching the number of available cores.

In [ ]:
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .addGrid(rf.maxBins, [32, 64, 128])
    .build()
)

evaluator = BinaryClassificationEvaluator(
    labelCol='readmission_30day', rawPredictionCol='raw_prediction', metricName='areaUnderROC'
)

cv = CrossValidator(
    estimator=pipeline, estimatorParamMaps=param_grid,
    evaluator=evaluator, numFolds=3, parallelism=2, seed=42
)

print(f'Grid size: {len(param_grid)} combos x 3 folds = {len(param_grid)*3} fits')
print('Training... (this may take several minutes)')

import time
start = time.time()
cv_model = cv.fit(train_df)
elapsed = time.time() - start
print(f'Done in {elapsed:.0f}s')

## Step 6: Evaluate on Test Set

We evaluate the best model on held-out test data with multiple metrics.

In [ ]:
predictions = cv_model.transform(test_df)

auc = evaluator.evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol='readmission_30day', predictionCol='prediction', metricName='f1').evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol='readmission_30day', predictionCol='prediction', metricName='accuracy').evaluate(predictions)

print(f'AUC-ROC:  {auc:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Accuracy: {accuracy:.4f}')

# Best hyperparameters
best_rf = cv_model.bestModel.stages[-1]
print(f'\nBest numTrees: {best_rf.getNumTrees}')
print(f'Best maxDepth: {best_rf.getOrDefault("maxDepth")}')
print(f'Best maxBins: {best_rf.getOrDefault("maxBins")}')

## Step 7: ROC Curve

The ROC curve plots the True Positive Rate vs False Positive Rate at various thresholds.
AUC closer to 1.0 means better discrimination between readmitted and non-readmitted patients.

In [ ]:
# Extract probabilities for ROC curve
from pyspark.ml.functions import vector_to_array

roc_data = (
    predictions
    .select('readmission_30day', vector_to_array('probability').alias('prob_array'))
    .withColumn('prob_positive', F.col('prob_array')[1])
    .select('readmission_30day', 'prob_positive')
    .toPandas()
)

from sklearn.metrics import roc_curve, auc as sk_auc
fpr, tpr, thresholds = roc_curve(roc_data['readmission_30day'], roc_data['prob_positive'])
roc_auc = sk_auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#1976D2', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#1976D2')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — 30-Day Readmission Prediction')
ax.legend(loc='lower right', fontsize=12)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

## Step 8: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

pred_pd = predictions.select('readmission_30day', 'prediction').toPandas()
cm = confusion_matrix(pred_pd['readmission_30day'], pred_pd['prediction'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — 30-Day Readmission')
plt.tight_layout()
plt.show()

## Step 9: Feature Importance

In [ ]:
importances = best_rf.featureImportances.toArray()
feature_names = numeric_cols + [f'{c}_ohe' for c in cat_cols]
while len(feature_names) < len(importances):
    feature_names.append(f'feature_{len(feature_names)}')
feature_names = feature_names[:len(importances)]

feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color='#1976D2')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 15 Feature Importances — RandomForest Readmission Model')
plt.tight_layout()
plt.show()

## Summary

| Component | Detail |
|-----------|--------|
| Model | RandomForestClassifier (PySpark MLlib) |
| Features | 20 numeric + 4 one-hot encoded categoricals |
| Tuning | CrossValidator, 27 param combos × 3 folds |
| Target | 30-day readmission (binary) |
| Evaluation | AUC-ROC, F1, Accuracy |

The model is saved as a complete `PipelineModel` — all preprocessing stages are
serialized together, ensuring no train/serve skew when deployed via FastAPI.

In [ ]:
spark.stop()
print('SparkSession stopped.')